# Baseline Eval — Mistral-7B-v0.3 on Docker Q&A

**Task #3** of the weekend plan. We run the **untrained** Mistral-7B over the 50 questions in `data/eval/eval.jsonl` and record its scores (ROUGE-1, ROUGE-L, Exact Match, GPT-4o-mini judge). These numbers are the "before" we will compare every fine-tuned run against.

**Runtime budget**: ~5–10 minutes on a free Colab T4.

**Before you run this notebook, make sure:**
1. Runtime → Change runtime type → **T4 GPU** is selected.
2. You've accepted the Mistral-7B license at https://huggingface.co/mistralai/Mistral-7B-v0.3 (one-time, click "Agree and access repository").
3. You've added three secrets in Colab's key-icon panel (left sidebar):
   - `HF_TOKEN` — your HuggingFace read token
   - `OPENAI_API_KEY` — your OpenAI key
   - `WANDB_API_KEY` — (optional for this notebook — only used during training)

## 1. Confirm GPU is available

If `nvidia-smi` fails or shows no T4, go to **Runtime → Change runtime type** and pick **T4 GPU**, then restart the runtime.

In [ ]:
!nvidia-smi

## 2. Clone the project from GitHub

Replace `<your-username>/LLM_Finetune` with your actual repo. If your repo is private, generate a fine-grained GitHub token with read access and use `https://<token>@github.com/<you>/LLM_Finetune.git` instead.

In [ ]:
!git clone https://github.com/<your-username>/LLM_Finetune.git
%cd LLM_Finetune

## 3. Install dependencies

Colab comes with most of what we need preinstalled. We additionally need:
- **`bitsandbytes`** — provides 4-bit quantization so Mistral-7B (~14 GB in bfloat16) fits into the T4's 15 GB.
- **`accelerate`** — handles `device_map="auto"` so the model places its layers on the GPU automatically.
- Our `requirements.txt` for `rouge_score`, `openai`, `python-dotenv`, etc.

Install takes ~30 seconds.

In [ ]:
!pip install -q -U bitsandbytes accelerate transformers
!pip install -q -r requirements.txt

## 4. Load secrets from Colab into environment variables

Our scripts read API keys from environment variables (via `python-dotenv`). On Colab, the recommended way to store secrets is the **Secrets** panel (🔑 icon in the left sidebar). We read them with `userdata.get()` and copy them into `os.environ` so the scripts pick them up.

This avoids ever putting raw tokens in the notebook cells (which would leak to git if you save the notebook).

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"]                 = userdata.get("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"]   = userdata.get("HF_TOKEN")  # alt name some libs use
os.environ["OPENAI_API_KEY"]           = userdata.get("OPENAI_API_KEY")

# Sanity check (prints whether each was found, not the actual value)
for k in ("HF_TOKEN", "OPENAI_API_KEY"):
    print(f"{k}: {'set' if os.environ.get(k) else 'MISSING'}")

## 5. Run the baseline eval

This will:
1. Download Mistral-7B-v0.3 weights (~14 GB, takes 2-3 minutes the first time).
2. Quantize to 4-bit and load onto the T4 (~5 GB VRAM after quant).
3. Generate an answer for each of the 50 eval questions.
4. Score each with ROUGE/Exact Match locally, then call GPT-4o-mini for the 1-5 judge.
5. Save `results/baseline/per_example.csv` and `results/baseline/results.json`.

**OpenAI cost**: ~$0.01 (50 short judge calls).

In [ ]:
!python scripts/02_baseline_eval.py

## 6. Inspect the results

We print the summary JSON and the top-5 weakest answers (lowest LLM-judge scores). The weak ones are previews of what fine-tuning needs to fix.

In [ ]:
import json, pandas as pd

with open("results/baseline/results.json") as f:
    summary = json.load(f)

print("=== Aggregate ===")
for k, v in summary["aggregate"].items():
    if v is not None and isinstance(v, float):
        print(f"  {k:15s} {v:.4f}")
    else:
        print(f"  {k:15s} {v}")

print("\n=== Worst 5 answers (lowest LLM-judge) ===")
df = pd.read_csv("results/baseline/per_example.csv")
worst = df.sort_values("judge_score").head(5)
for _, row in worst.iterrows():
    print(f"\n[{row['id']}] judge={row['judge_score']}  rougeL={row['rougeL']:.2f}")
    print(f"  Q: {row['question']}")
    print(f"  Model: {str(row['prediction'])[:300]}...")
    print(f"  Ref:   {str(row['reference'])[:300]}...")

## 7. Commit the results back

We need `results/baseline/results.json` back on your local machine so the Sunday "compile results" step can include it. Two options:

**Option A — Commit from Colab** (quick, requires GitHub token with write access):
```bash
!git config user.email "you@example.com"
!git config user.name "Your Name"
!git add results/baseline/results.json
!git commit -m "Add baseline eval results"
!git push https://<your-pat>@github.com/<you>/LLM_Finetune.git
```

**Option B — Download manually** (no git push needed):
Use the Files panel in Colab (left sidebar), navigate to `LLM_Finetune/results/baseline/`, right-click → Download both `results.json` and `per_example.csv`, then move them into your local `results/baseline/` folder.